# 04 – KMeans Exploration

**Entrée** : `data/processed/attrition_with_avg_hours.csv`

**Export unique** : `data/processed/kmeans_clusters.csv`

## 1. Imports et paramètres

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

# === PARAMETRES MODIFIABLES ===
K_RANGE = range(2, 11)          # Plage k pour analyse silhouette
K_CANDIDATES = [2, 3, 4]        # k candidats pour silhouette plots
K_OVERRIDE = None               # Forcer k final (None = auto)
RANDOM_STATE = 42
N_INIT = 50

# Chemins
DATA_PATH = Path("..", "data", "processed", "attrition_with_avg_hours.csv")
FIG_DIR = Path("..", "reports", "figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

## 2. Chargement et sélection colonnes numériques

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape original: {df.shape}")

# Colonnes à exclure
EXCLUDE_COLS = ["EmployeeID", "Attrition", "cluster"]

# Sélection colonnes numériques uniquement
num_cols = df.select_dtypes(include=["number"]).columns.tolist()
X_cols = [c for c in num_cols if c not in EXCLUDE_COLS]
X_num = df[X_cols].copy()

print(f"\nColonnes retenues ({len(X_cols)}):")
print(X_cols)
print(f"\nX_num.shape: {X_num.shape}")

## 3. Standardisation

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_num)

print(f"X_scaled.shape: {X_scaled.shape}")
print(f"NaN dans X_scaled: {np.isnan(X_scaled).sum()}")

## 4. Analyse silhouette multi-k

In [ ]:
scores = []
inertias = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    scores.append(sil)
    inertias.append(km.inertia_)
    print(f"k={k:2d}  silhouette={sil:.4f}  inertia={km.inertia_:,.0f}")

k_best_auto = list(K_RANGE)[np.argmax(scores)]
print(f"\nk_best_auto (argmax silhouette) = {k_best_auto}")

### 4.1 Courbe silhouette vs k

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), scores, "bo-", linewidth=2, markersize=8)
ax.axhline(y=max(scores), color="r", linestyle="--", alpha=0.5)
ax.set_xlabel("k")
ax.set_ylabel("Silhouette moyenne")
ax.set_title("Score silhouette vs k")
ax.set_xticks(list(K_RANGE))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_silhouette_vs_k.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'kmeans_silhouette_vs_k.png'}")

### 4.2 Courbe Elbow (inertie)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), inertias, "go-", linewidth=2, markersize=8)
ax.set_xlabel("k")
ax.set_ylabel("Inertie")
ax.set_title("Méthode du coude (Elbow)")
ax.set_xticks(list(K_RANGE))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_elbow.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'kmeans_elbow.png'}")

## 5. Silhouette analysis (style scikit-learn)

In [ ]:
# PCA 2D (une seule fois)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
var_pc1 = pca.explained_variance_ratio_[0]
var_pc2 = pca.explained_variance_ratio_[1]
print(f"Variance expliquée: PC1={var_pc1:.1%}, PC2={var_pc2:.1%}")

In [ ]:
for n_clusters in K_CANDIDATES:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # KMeans fit
    clusterer = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=N_INIT)
    cluster_labels = clusterer.fit_predict(X_scaled)
    
    # Silhouette moyenne
    silhouette_avg = silhouette_score(X_scaled, cluster_labels)
    sample_silhouette_values = silhouette_samples(X_scaled, cluster_labels)
    
    # === AX1: Silhouette plot ===
    ax1.set_xlim([-0.1, 1])
    ax1.set_ylim([0, len(X_scaled) + (n_clusters + 1) * 10])
    
    y_lower = 10
    for i in range(n_clusters):
        ith_cluster_silhouette_values = sample_silhouette_values[cluster_labels == i]
        ith_cluster_silhouette_values.sort()
        size_cluster_i = ith_cluster_silhouette_values.shape[0]
        y_upper = y_lower + size_cluster_i
        
        color = cm.nipy_spectral(float(i) / n_clusters)
        ax1.fill_betweenx(
            np.arange(y_lower, y_upper),
            0,
            ith_cluster_silhouette_values,
            facecolor=color,
            edgecolor=color,
            alpha=0.7,
        )
        ax1.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))
        y_lower = y_upper + 10
    
    ax1.set_title(f"Silhouette plot (k={n_clusters})")
    ax1.set_xlabel("Coefficient silhouette")
    ax1.set_ylabel("Cluster")
    ax1.axvline(x=silhouette_avg, color="red", linestyle="--", label=f"Moyenne={silhouette_avg:.3f}")
    ax1.set_yticks([])
    ax1.legend(loc="upper right")
    
    # === AX2: PCA scatter ===
    colors = cm.nipy_spectral(cluster_labels.astype(float) / n_clusters)
    ax2.scatter(X_pca[:, 0], X_pca[:, 1], marker=".", s=30, lw=0, alpha=0.7, c=colors)
    
    # Centres projetés
    centers_pca = pca.transform(clusterer.cluster_centers_)
    ax2.scatter(centers_pca[:, 0], centers_pca[:, 1], marker="o", c="white", alpha=1, s=200, edgecolor="k")
    for i, c in enumerate(centers_pca):
        ax2.scatter(c[0], c[1], marker=f"${i}$", alpha=1, s=50, edgecolor="k")
    
    ax2.set_title(f"PCA 2D (PC1={var_pc1:.1%}, PC2={var_pc2:.1%})")
    ax2.set_xlabel("PC1")
    ax2.set_ylabel("PC2")
    
    fig.suptitle(f"Silhouette Analysis k={n_clusters}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    fig_path = FIG_DIR / f"kmeans_silhouette_analysis_k{n_clusters}.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"k={n_clusters}: silhouette={silhouette_avg:.4f} | Saved: {fig_path}")

## 6. Choix final de k

In [ ]:
k_final = K_OVERRIDE if K_OVERRIDE is not None else k_best_auto

print(f"k_best_auto (argmax silhouette): {k_best_auto}")
print(f"K_OVERRIDE: {K_OVERRIDE}")
print(f"k_final: {k_final}")

# Fit final
kmeans_final = KMeans(n_clusters=k_final, random_state=RANDOM_STATE, n_init=N_INIT)
labels_final = kmeans_final.fit_predict(X_scaled)
sil_final = silhouette_score(X_scaled, labels_final)

print(f"\nSilhouette finale: {sil_final:.4f}")
print(f"Tailles clusters: {np.bincount(labels_final).tolist()}")

### 6.1 PCA final

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

colors = cm.nipy_spectral(labels_final.astype(float) / k_final)
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colors, marker=".", s=30, alpha=0.7)

centers_pca = pca.transform(kmeans_final.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1], marker="o", c="white", s=200, edgecolor="k")
for i, c in enumerate(centers_pca):
    ax.scatter(c[0], c[1], marker=f"${i}$", s=50, edgecolor="k")

ax.set_xlabel(f"PC1 ({var_pc1:.1%})")
ax.set_ylabel(f"PC2 ({var_pc2:.1%})")
ax.set_title(f"KMeans k={k_final} | Silhouette={sil_final:.4f}")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / "kmeans_pca_final.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'kmeans_pca_final.png'}")

### 6.2 Features les plus influentes

In [ ]:
cluster_means = pd.DataFrame(
    scaler.inverse_transform(kmeans_final.cluster_centers_),
    columns=X_cols
)
cluster_means.index.name = "cluster"

df_with_labels = X_num.copy()
df_with_labels['cluster'] = labels_final

# CALCUL DE LA VARIANCE (η² - eta-squared)
# η² = proportion de la variance totale expliquée par les différences entre clusters
# Formule: η² = SS_between / SS_total
# Plus η² est proche de 1, mieux la variable sépare les clusters

variance_explained = {}
for col in X_cols:
    # Variance totale (SS_total)
    grand_mean = df_with_labels[col].mean()
    ss_total = ((df_with_labels[col] - grand_mean) ** 2).sum()
    
    # Variance inter-cluster (SS_between)
    ss_between = 0
    for cluster_id in range(k_final):
        cluster_data = df_with_labels[df_with_labels['cluster'] == cluster_id][col]
        cluster_mean = cluster_data.mean()
        cluster_size = len(cluster_data)
        ss_between += cluster_size * (cluster_mean - grand_mean) ** 2
    
    eta_squared = ss_between / ss_total if ss_total > 0 else 0
    variance_explained[col] = eta_squared

variance_explained_series = pd.Series(variance_explained).sort_values(ascending=False)

top_n = 15
top_features = variance_explained_series.head(top_n)

print(f"=== TOP {top_n} FEATURES QUI SÉPARENT LE MIEUX LES CLUSTERS ===")
print("(basé sur η² - proportion de variance expliquée par les clusters)")
print("→ η² proche de 1 = excellente séparation | η² proche de 0 = clusters se chevauchent\n")
for i, (feat, eta2) in enumerate(top_features.items(), 1):
    quality = "★★★" if eta2 > 0.5 else "★★" if eta2 > 0.2 else "★"
    print(f"{i:2d}. {feat:30s} | η² = {eta2:.4f} ({eta2*100:.2f}%) {quality}")

# Afficher les moyennes par cluster (top features)
print(f"\n\n=== MOYENNES PAR CLUSTER (TOP {top_n} FEATURES) ===")
top_feature_names = top_features.index.tolist()
cluster_comparison = cluster_means[top_feature_names].T
cluster_comparison.columns = [f"Cluster {i}" for i in range(k_final)]
print(cluster_comparison.round(2))

print(f"\n\n=== STATISTIQUES DE SÉPARATION ===")
print(f"Variables avec excellente séparation (η² > 0.5): {(variance_explained_series > 0.5).sum()}")
print(f"Variables avec bonne séparation (0.2 < η² ≤ 0.5): {((variance_explained_series > 0.2) & (variance_explained_series <= 0.5)).sum()}")
print(f"Variables avec faible séparation (η² ≤ 0.2): {(variance_explained_series <= 0.2).sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
top_features.plot(kind="barh", ax=ax, color="steelblue")
ax.set_xlabel("η² (Proportion de variance expliquée)")
ax.set_ylabel("Feature")
ax.set_title(f"Top {top_n} features qui séparent le mieux les clusters")
ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='Seuil excellente séparation (0.5)')
ax.axvline(x=0.2, color='orange', linestyle='--', alpha=0.5, label='Seuil bonne séparation (0.2)')
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / "top_influential_features.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'top_influential_features.png'}")

In [ ]:
cluster_means_normalized = cluster_means[top_feature_names].copy()
for col in top_feature_names:
    min_val = cluster_means_normalized[col].min()
    max_val = cluster_means_normalized[col].max()
    if max_val > min_val:
        cluster_means_normalized[col] = (cluster_means_normalized[col] - min_val) / (max_val - min_val)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cluster_means_normalized.T, 
    annot=cluster_means[top_feature_names].T.round(1), 
    fmt=".1f",
    cmap="RdYlGn",
    cbar_kws={"label": "Valeur normalisée (0-1)"},
    ax=ax
)
ax.set_xlabel("Cluster")
ax.set_ylabel("Feature")
ax.set_title(f"Profil des clusters : Top {top_n} features")
ax.set_xticklabels([f"Cluster {i}" for i in range(k_final)])
plt.tight_layout()
plt.savefig(FIG_DIR / "cluster_profiles_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'cluster_profiles_heatmap.png'}")

## 7. Export

In [ ]:
# Ajouter cluster au df original
df["cluster"] = labels_final

output_path = Path("..", "data", "processed", "kmeans_clusters.csv")
df.to_csv(output_path, index=False)

print("=== EXPORT CHECK ===")
print(f"Path: {output_path}")
print(f"Shape: {df.shape}")
print(f"NaN total: {df.isna().sum().sum()}")
print("\ncluster value_counts():")
print(df["cluster"].value_counts().sort_index())

## 8. Validation

In [ ]:
# Assertions obligatoires
assert X_num.shape[0] == 4410, f"X_num rows: {X_num.shape[0]} != 4410"
assert np.isnan(X_scaled).sum() == 0, "NaN in X_scaled"
assert "EmployeeID" not in X_num.columns, "EmployeeID in X_num"
assert "Attrition" not in X_num.columns, "Attrition in X_num"

print("Toutes les assertions passent.")